Import numpy and matplotlib

In [12]:
import numpy as np
import matplotlib.pyplot as plt

Load data

In [13]:
data = np.genfromtxt(
    'Iris.csv',
    delimiter=',',
    dtype=str,
    skip_header=1
)

seperate datas as values and labels(answer)

In [14]:
X = data[:,1:5].astype(float)
labels = data[:,5]


maping labels

In [15]:
species_map = {
    'Iris-setosa': [1, 0, 0],
    'Iris-versicolor': [0, 1, 0],
    'Iris-virginica': [0, 0, 1]
}
Y = np.array([species_map[label] for label in labels])


Normalize the data


In [16]:
X_min = X.min(axis=0)
X_max = X.max(axis=0)

X = (X - X_min) / (X_max - X_min)



Shuffle the data and creating the test sample of 120 datas

In [17]:
combined = np.hstack((X, Y))
np.random.shuffle(combined)

X = combined[:, :4]
Y = combined[:, 4:]

X_train = X[:120]
Y_train = Y[:120]

X_test = X[120:]
Y_test = Y[120:]

Initialize the random weights and biases for hidden layer 1, 2 and the output layer

In [18]:
np.random.seed(42)

W1 = np.random.randn(10, 4)
b1 = np.random.randn(10, 1)

W2 = np.random.randn(5, 10)
b2 = np.random.randn(5, 1)

W3 = np.random.randn(3, 5)
b3 = np.random.randn(3, 1)

defining the relu and softmax functions

In [19]:
def relu(z):
    return np.maximum(0, z)

def softmax(z):
    exp_z = np.exp(z - np.max(z))
    return exp_z / np.sum(exp_z, axis=0, keepdims=True)

def relu_derivative(z):
    return (z > 0).astype(float)

defining the forward pass function

In [20]:
def forward_pass(x):

    Z1 = np.dot(W1, x) + b1
    A1 = relu(Z1)

    Z2 = np.dot(W2, A1) + b2
    A2 = relu(Z2)

    Z3 = np.dot(W3, A2) + b3
    A3 = softmax(Z3)

    cache=(x,Z1,A1,Z2,A2,Z3,A3)

    return A3,cache

Defining Cross Entropy

In [21]:
def cross_entropy_loss(y_true, y_pred):
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    loss = -np.sum(y_true * np.log(y_pred))
    return loss

Defining backward pass for finding gradients of all parameters

In [22]:
def backward_pass(y, cache):

    x, Z1, A1, Z2, A2, Z3, A3 = cache

    dZ3 = A3 - y
    dW3 = np.dot(dZ3, A2.T)
    db3 = dZ3

    dA2 = np.dot(W3.T, dZ3)
    dZ2 = dA2 * relu_derivative(Z2)

    dW2 = np.dot(dZ2, A1.T)
    db2 = dZ2

    dA1 = np.dot(W2.T, dZ2)
    dZ1 = dA1 * relu_derivative(Z1)

    dW1 = np.dot(dZ1, x.T)
    db1 = dZ1

    return dW1, db1, dW2, db2, dW3, db3

Defining Parameter update function

In [23]:
learning_rate = 0.01
def update_parameters(dW1, db1, dW2, db2, dW3, db3):
    global W1, b1, W2, b2, W3, b3

    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    W3 -= learning_rate * dW3
    b3 -= learning_rate * db3

Training the model

In [24]:
epochs = 100
for epoch in range(epochs):

    total_loss = 0

    for i in range(len(X_train)):

        x = X_train[i].reshape(4, 1)
        y = Y_train[i].reshape(3, 1)

        prediction, cache = forward_pass(x)

        loss = cross_entropy_loss(y, prediction)
        total_loss += loss

        dW1, db1, dW2, db2, dW3, db3 = backward_pass(y, cache)

        update_parameters(dW1, db1, dW2, db2, dW3, db3)

In [25]:
correct=0
def predict(x):
    prediction, _ = forward_pass(x)
    return np.argmax(prediction)

for i in range(len(X_test)):

    x = X_test[i].reshape(4, 1)

    predicted = predict(x)
    actual = np.argmax(Y_test[i])

    if predicted == actual:
        correct += 1
accuracy = (correct / len(X_test)) * 100
print("Accuracy:", accuracy, "%")


Accuracy: 96.66666666666667 %
